# Estadística, tiempos de espera y reproducibilidad

Analiza conjuntos de trayectorias, distribuciones de tiempos de permanencia, variabilidad y semillas aleatorias.

> **Nota científica:** ProteinMotorSim es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

In [ ]:
from pathlib import Path
import sys

# Busca la raíz del proyecto para poder importar src/protein_motor_sim.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "protein_motor_sim").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No encuentro la carpeta src/protein_motor_sim. "
        "Abre JupyterLab desde la raíz de ProteinMotorSim o instala el paquete "
        "con python -m pip install -e ."
    )

print(f"Proyecto detectado en: {PROJECT_ROOT}")

## 1. Objetivos

Este notebook estudia la variabilidad entre motores simulados, los tiempos de espera y la reproducibilidad. Las realizaciones son independientes y usan semillas derivadas de una semilla maestra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.simulation.ensemble import run_ensemble
from protein_motor_sim.analysis.statistics import summarize_trajectories

## 2. Simular un conjunto de motores

Una trayectoria individual no caracteriza por sí sola el comportamiento estadístico. Ejecutamos muchas realizaciones bajo las mismas condiciones.

In [ ]:
motor = Kinesin1()
environment = Environment(atp_concentration=1e-3, external_force_N=0.0, temperature_K=310.0)
simulator = GillespieSimulator()
ensemble = run_ensemble(
    simulator, motor, environment, t_max=0.5, n_runs=200, seed=2026
)
summary = summarize_trajectories(ensemble.trajectories)
summary

In [ ]:
velocities = np.array([tr.mean_velocity_nm_s for tr in ensemble.trajectories])
net_displacements = np.array([tr.displacement_nm for tr in ensemble.trajectories])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(velocities, bins=25, edgecolor="white")
axes[0].set_xlabel("Velocidad media neta (nm/s)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de velocidades")
axes[1].hist(net_displacements, bins=25, edgecolor="white")
axes[1].set_xlabel("Desplazamiento neto (nm)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de desplazamientos")
plt.tight_layout()
plt.show()

## 3. Distribución de tiempos de espera en un estado

Para una red de tasas constantes dentro de un estado dado, el tiempo de permanencia tiene distribución exponencial con parámetro igual a la suma de las tasas salientes. Extraemos los tiempos de eventos que salen de `bound_apo`. Se excluyen los intervalos censurados al final de observación.

In [ ]:
dwell_times = []
for tr in ensemble.trajectories:
    for i in range(1, len(tr.times_s)):
        source_state = tr.states[i - 1]
        event = tr.events[i]
        if source_state == "bound_apo" and event != "observation_end":
            dwell_times.append(tr.times_s[i] - tr.times_s[i - 1])

dwell_times = np.asarray(dwell_times)
network = motor.build_network()
total_exit_rate = sum(rate for _, rate in network.rates("bound_apo", environment))
print(f"Intervalos observados: {len(dwell_times)}")
print(f"Tasa total teórica de salida: {total_exit_rate:.3f} s^-1")
print(f"Media teórica: {1 / total_exit_rate:.5f} s")
print(f"Media muestral: {dwell_times.mean():.5f} s")

In [ ]:
if len(dwell_times):
    tau = np.linspace(0, np.quantile(dwell_times, 0.99), 250)
    pdf = total_exit_rate * np.exp(-total_exit_rate * tau)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(dwell_times, bins=30, density=True, alpha=0.65, label="Simulación")
    ax.plot(tau, pdf, linewidth=2, label="Exponencial teórica")
    ax.set_xlabel("Tiempo de permanencia en bound_apo (s)")
    ax.set_ylabel("Densidad de probabilidad (s$^{-1}$)")
    ax.set_title("Distribución de tiempos de espera")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.show()

## 4. Reproducibilidad

La misma semilla, el mismo modelo y las mismas condiciones deben generar la misma trayectoria. Cambiar la semilla produce otra realización posible.

In [ ]:
first = simulator.run(motor, environment, t_max=0.5, seed=77)
repeat = simulator.run(motor, environment, t_max=0.5, seed=77)
other = simulator.run(motor, environment, t_max=0.5, seed=78)

print("Misma semilla: tiempos idénticos =", first.times_s == repeat.times_s)
print("Misma semilla: eventos idénticos =", first.events == repeat.events)
print("Semillas distintas: eventos idénticos =", first.events == other.events)

## 5. ATP consumido y pasos

El modelo registra una molécula de ATP en cada transición `atp_hydrolysis`. Cuenta también los eventos forward y backward. En esta red simplificada la hidrólisis y el paso no están estrictamente acoplados uno a uno: el desprendimiento puede ocurrir entre ambos. Esta limitación es una razón para mejorar posteriormente el ciclo mecanoquímico.

In [ ]:
records = []
for i, tr in enumerate(ensemble.trajectories):
    records.append({
        "trajectory": i,
        "atp_hydrolyzed": tr.atp_consumed_total,
        "forward_steps": tr.events.count("forward_step"),
        "backward_steps": tr.events.count("backward_step"),
        "detached": "detachment" in tr.events,
        "net_displacement_nm": tr.displacement_nm,
    })
summary_df = pd.DataFrame(records)
display(summary_df.head())
print("Fracción desprendida:", summary_df["detached"].mean())
print("ATP promedio registrado:", summary_df["atp_hydrolyzed"].mean())

## 6. Interpretación

La dispersión entre trayectorias es parte del resultado, no un error numérico. Para comparar con un experimento será necesario definir el observable de la misma manera, considerar trayectorias censuradas y utilizar parámetros cinéticos medidos o ajustados.